In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.olist_bronce;
CREATE SCHEMA IF NOT EXISTS workspace.olist_plata;
CREATE SCHEMA IF NOT EXISTS workspace.olist_oro;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.olist_bronce.archivos;

In [0]:
import os
import pandas as pd
from pyspark.sql import functions as F

RUTA = "/Volumes/workspace/olist_bronce/archivos"
CATALOGO = "workspace.olist_bronce"

# 1. Lista solo los .csv del volumen
ARCHIVOS = [a for a in os.listdir(RUTA) if a.endswith(".csv")]
print(f"Archivos encontrados: {len(ARCHIVOS)}")

for archivo in ARCHIVOS:
    # 2. Nombre de la tabla: olist_orders_dataset.csv -> orders
    tabla = archivo.replace("olist_", "").replace("_dataset", "").replace(".csv", "")

    # 3. Leer todo como texto (bronce = crudo, sin interpretar tipos)
    df = pd.read_csv(f"{RUTA}/{archivo}", dtype=str)

    # 4. Los vacíos de pandas (NaN) pasan a None para que Spark los lea como NULL
    df = df.astype(object).where(df.notna(), None)

    # 5. A Spark, con columnas de trazabilidad, y se guarda como tabla Delta
    sdf = (spark.createDataFrame(df)
                .withColumn("archivo_origen", F.lit(archivo))
                .withColumn("fecha_ingesta", F.current_timestamp()))

    (sdf.write.mode("overwrite")
              .option("overwriteSchema", "true")
              .saveAsTable(f"{CATALOGO}.{tabla}"))

    print(f"Se cargó {CATALOGO}.{tabla}: {df.shape[0]} filas")

In [0]:
%sql
SHOW TABLES IN workspace.olist_bronce;